# weight-sparsity on Google Colab

Train a TinyStories LM with differentiable weight sparsity
([LTP](https://arxiv.org/abs/2003.00075) / [Continuous Sparsification](https://arxiv.org/abs/1912.04427) /
TopK + soft gate).

**Runtime → Change runtime type → GPU** (T4 is fine for the small config; use an
A100/L4 for the ~150M one).

Everything that has to survive a disconnect — the tokenised dataset and the
checkpoints — lives on Google Drive under `MyDrive/weight-sparsity/`. The token
`.bin` files are also cached on the local disk, because reading training batches
straight off Drive is slow.

In [ ]:
!nvidia-smi

## 1. Mount Drive and pick the paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# --- Google Drive paths (persist across sessions) ---------------------------
PROJECT     = '/content/drive/MyDrive/weight-sparsity'
DRIVE_DATA  = f'{PROJECT}/data/tinystories'   # cached token stream
RUNS_DIR    = f'{PROJECT}/runs'               # checkpoints + metrics

# --- local scratch (fast, wiped when the runtime dies) ----------------------
REPO_DIR    = '/content/weight-sparsity'
LOCAL_DATA  = '/content/data/tinystories'     # what training actually reads

for p in (PROJECT, DRIVE_DATA, RUNS_DIR, LOCAL_DATA):
    os.makedirs(p, exist_ok=True)

print('drive project :', PROJECT)
print('runs          :', RUNS_DIR)

## 2. Clone the repo and install

In [ ]:
REPO_URL = 'https://github.com/labofdoubt/weight-sparsity.git'

if not os.path.exists(REPO_DIR):
    !git clone -q $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull -q

%cd $REPO_DIR
!git log --oneline -1

In [ ]:
# Colab already ships torch, so install the package without pulling it again.
!pip install -q datasets transformers tokenizers pyyaml tqdm
!pip install -q -e . --no-deps

import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 3. Prepare the data (once)

Downloads TinyStories, tokenises it with the GPT-Neo tokenizer and writes flat
`uint16` streams. Takes ~10-15 minutes the first time; afterwards it is copied
from Drive in a few seconds.

Set `TOKENIZER = 'bpe'` for an 8k byte-level BPE trained on TinyStories itself —
that shrinks the embedding matrix from 39M to 6M parameters, so more of the
budget sits in the transformer body (and in the maskable MLP weights).

In [ ]:
TOKENIZER = 'gpt_neo'   # or 'bpe'

drive_ready = os.path.exists(f'{DRIVE_DATA}/meta.json')

if drive_ready:
    print('copying the cached dataset from Drive ...')
    !cp -u $DRIVE_DATA/*.bin $DRIVE_DATA/meta.json $LOCAL_DATA/
else:
    print('preparing the dataset from scratch ...')
    cmd = (f'python -m wsparse.data --config configs/dense.yaml '
           f'--data.data_dir={LOCAL_DATA} --data.tokenizer={TOKENIZER} '
           f'--data.tokenizer_path={LOCAL_DATA}/tokenizer --data.num_proc=2')
    !{cmd}
    print('caching it to Drive for next time ...')
    !cp -u $LOCAL_DATA/*.bin $LOCAL_DATA/meta.json $DRIVE_DATA/
    !cp -ru $LOCAL_DATA/tokenizer $DRIVE_DATA/ 2>/dev/null || true

!ls -lh $LOCAL_DATA
!cat $LOCAL_DATA/meta.json

## 4. Choose the run

`CONFIG` picks the recipe, the overrides after it are applied on top. Config
files live in `configs/`:

| config | what it does |
| --- | --- |
| `dense.yaml` | dense baseline |
| `ltp.yaml` | learned per-layer threshold on `w²`, smooth-L0 penalty |
| `cs.yaml` | per-weight gates `s`, smooth-L0 penalty |
| `topk.yaml` | **hard TopK forward support, wider Top-(k+j) backward support** |
| `topk_soft_l0.yaml` | same, plus the soft-L0 penalty over Top-(k+j) |
| `ltp_target.yaml` / `cs_target.yaml` | same, but driven to a target density |
| `ltp_150m.yaml` / `cs_150m.yaml` / `topk_150m.yaml` | the ~152M-parameter model |
The default below is **TopK + soft gate**: `k` weights per tensor stay active in
the forward pass (a hard FLOP budget — no penalty has to negotiate it), while
the backward pass runs over the wider Top-(k+j) support, so the extra `j`
candidates keep learning and can enter TopK later. Set `--sparsity.method=ltp`
or `cs` to fall back to the threshold/gate methods, which is what the other
configs in the table do.

Model size comes from `configs/model/{tiny,small,medium,large}.yaml` and can be
swapped inline with `--model.n_layers=... --model.d_model=...`.

In [ ]:
CONFIG   = 'configs/topk.yaml'
RUN_NAME = 'topk_small_colab'

OVERRIDES = [
    f'--data.data_dir={LOCAL_DATA}',
    f'--train.out_dir={RUNS_DIR}',
    f'--train.run_name={RUN_NAME}',
    '--train.max_steps=20000',
    '--train.batch_size=64',
    '--train.micro_batch_size=16',     # lower this if you hit OOM
    '--train.lr=6e-4',
    '--train.validate_every_steps=500',
    '--train.checkpoint_every_steps=1000',
    '--train.sample_every_steps=2000',
    # --- sparsity handles -------------------------------------------------
    '--sparsity.enabled=true',
    '--sparsity.method=topk',           # topk | ltp | cs
    '--sparsity.targets=["mlp"]',       # ["mlp","attn"] to sparsify attention too
    # forward support: k active weights per group (a fraction, or a count >= 1)
    '--sparsity.k=0.1',
    # backward-only support: j extra candidates that get gradients for w and s
    # but contribute nothing to the forward pass
    '--sparsity.j=0.05',
    '--sparsity.topk_groups=tensor',    # tensor | row | block
    # '--sparsity.topk_groups=block', '--sparsity.topk_block_size=4', '--sparsity.k=2',   # 2:4
    '--sparsity.w_grad_support=topk_j', # topk -> restrict w gradients to TopK
    # inverse temperature: on its own it pins beta constant; with a schedule it
    # is the starting point of the anneal towards beta_end
    '--sparsity.inverse_temperature=4.0',
    '--sparsity.inverse_temperature_schedule=exponential',
    '--sparsity.beta_end=200',
    '--sparsity.s_init_mode=magnitude', # initial TopK = the top-k weights by |w|
    '--sparsity.s_init=1.0',
    '--sparsity.mask_lr=1e-2',
    '--sparsity.mask_grad_clip=1.0',    # dL/ds carries a factor beta
    # soft L0 over Top-(k+j): shrinks gates *inside* the TopK budget.
    # These are per-weight coefficients and are NOT normalised, unlike l0_coef.
    # '--sparsity.soft_l0_enabled=true',
    # '--sparsity.soft_l0_lambda_topk=1e-7',
    # '--sparsity.soft_l0_lambda_explore=1e-8',
]
OVERRIDE_STR = ' '.join(OVERRIDES)
print(OVERRIDE_STR)

In [ ]:
!python scripts/model_summary.py --config $CONFIG $OVERRIDE_STR

## 5. Train

The log line shows the loss, the inverse temperature `beta`, the soft and hard
density, and `trans` — the fraction of weights still inside the sigmoid's
transition band. If `trans` collapses to 0 early, beta is annealing too fast and
the mask freezes before it has finished sorting the weights.

For `topk` there are two more fields. `gate` is the mean `σ(β·s)` over the
selected weights, and `turn` is the fraction of TopK that changed at the last
re-selection — **if `turn` sits at 0 from early on, `j` is buying you nothing**,
because the exploratory candidates never overtake an incumbent. The startup
banner also prints the resolved `k`/`j` and the forward density, which for
`topk` is a hard budget rather than something the penalties drive.

In [ ]:
!python -m wsparse.train --config $CONFIG $OVERRIDE_STR

### Resuming after a disconnect

Checkpoints go to Drive, so re-run the cell above with `--train.resume=auto`
appended and it picks up from the last checkpoint.

In [ ]:
# !python -m wsparse.train --config $CONFIG $OVERRIDE_STR --train.resume=auto

## 6. Curves

In [ ]:
import json
import matplotlib.pyplot as plt

path = f'{RUNS_DIR}/{RUN_NAME}/metrics.jsonl'
records = [json.loads(l) for l in open(path)]

def series(key):
    xs = [(r['step'], r[key]) for r in records if key in r]
    return [x for x, _ in xs], [y for _, y in xs]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

s, ce = series('train/ce')
axes[0].plot(s, ce, label='train')
s, v = series('val/ce')
axes[0].plot(s, v, 'o-', label='val (soft)')
s, vh = series('val_hard/ce')
if s:
    axes[0].plot(s, vh, 's--', label='val (hard mask)')
axes[0].set_xlabel('step'); axes[0].set_ylabel('cross-entropy'); axes[0].legend(); axes[0].grid(alpha=.3)

s, d = series('sparsity/density_soft')
if s:
    axes[1].plot(s, d, label='soft (smooth L0)')
    s, dh = series('sparsity/density_hard')
    axes[1].plot(s, dh, label='hard')
    # topk: the TopK budget is a hard ceiling the other two can only sit under
    s, dk = series('sparsity/density_topk')
    if s:
        axes[1].plot(s, dk, 'k:', label='topk budget (k/N)')
    axes[1].set_ylim(0, 1.02)
axes[1].set_xlabel('step'); axes[1].set_ylabel('density'); axes[1].legend(); axes[1].grid(alpha=.3)

s, b = series('sparsity/beta')
if s:
    axes[2].semilogy(s, b, color='tab:red')
    ax2 = axes[2].twinx()
    s2, t = series('sparsity/transition_frac')
    ax2.plot(s2, t, color='tab:gray', alpha=.6, label='transition frac')
    # topk: TopK churn per step. Flat 0 means the j candidates never win.
    s3, turn = series('sparsity/turnover')
    if s3:
        ax2.plot(s3, turn, color='tab:green', alpha=.8, label='topk turnover')
        ax2.legend(loc='center right', fontsize=8)
    ax2.set_ylabel('transition fraction / turnover')
axes[2].set_xlabel('step'); axes[2].set_ylabel('beta (log)'); axes[2].grid(alpha=.3)

plt.tight_layout(); plt.show()

## 7. Sample from the checkpoint

`--hard` runs the network with binary masks, i.e. the actually pruned model.

In [ ]:
CKPT = f'{RUNS_DIR}/{RUN_NAME}/latest.pt'
!python scripts/generate.py --ckpt "$CKPT" --prompt "Once upon a time" --tokens 200
print()
!python scripts/generate.py --ckpt "$CKPT" --prompt "Once upon a time" --tokens 200 --hard

## 8. Per-layer density

Which layers the method decided to prune hardest. For `topk` this is capped by
`k` in every layer by construction, so what it shows is how far each layer's
gates fell *below* its budget.

In [ ]:
last = [r for r in records if any(k.startswith('layer_') for k in r)][-1]
for k, v in sorted((k, v) for k, v in last.items() if k.startswith('layer_')):
    print(f'{k[6:]:<28} {v:.4f}')